# دفتر الطالب | تشغيل دفعة واحدة
بعد ضبط الخيارات اختر **Runtime → Run all**. المعاينة لا تحتاج أي حساب للنشر. YouTube يرفع خاصًا ما لم تغيّر مستوى الظهور؛ Facebook ينشر مباشرة. لدفعة محفوظة قابلة للاستئناف اختر Drive (يتطلب موافقتك). لا تشغّل جلستين على الدفعة نفسها. لا توجد محاولة لمنع انتهاء جلسة Colab.

In [ ]:
#@title إعدادات الدفعة (احفظ نسخة من الدفتر للاحتفاظ باختياراتك)
count = 1 #@param {type:"slider", min:1, max:5, step:1}
platform = 'preview' #@param ['preview', 'youtube', 'facebook', 'both']
visibility = 'private' #@param ['private', 'unlisted', 'public']
voice = False #@param {type:"boolean"}
pexels = False #@param {type:"boolean"}
save_to_drive = False #@param {type:"boolean"}
assert platform in ('preview','youtube','facebook','both') and 1 <= count <= 5

In [ ]:
#@title تحميل نسخة البرنامج المحددة وتثبيت الاعتماديات
from pathlib import Path
import os, sys, subprocess, tempfile, shutil
from datetime import datetime, timezone
from google.colab import drive, userdata
from IPython.display import Video, display
# Download a fixed revision, not arbitrary code from a moving branch.
rev = '96e7c1d78b337976fa767d3555d72c9a0279fb89'
root = Path(tempfile.mkdtemp(prefix='arara-'))
subprocess.run(['git','init','-q',str(root)], check=True)
def git(*args):
    subprocess.run(['git','-C',str(root),*args],check=True)
git('fetch','-q','--depth=1','https://github.com/Wynm/ARARA.git',rev)
git('checkout','-q','--detach',rev)
req = 'requirements.txt' if platform == 'preview' and not voice else 'requirements-extra.txt'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(root/req)],check=True)
base = Path('/content/arara-output')
if save_to_drive:
    drive.mount('/content/drive')
    base = Path('/content/drive/MyDrive/ARARA')
base.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title إنتاج وحفظ ثم نشر كل درس على المنصات المختارة
# Each UTC day has a finite batch. Re-running the same settings reuses its videos.
day = datetime.now(timezone.utc).date().isoformat()
batch = base / f'{day}-voice{int(voice)}-photo{int(pexels)}'
batch.mkdir(exist_ok=True)
targets = ['youtube','facebook'] if platform == 'both' else ([] if platform=='preview' else [platform])
def env_for(names):
    return {**os.environ, **{name: userdata.get(name) for name in names}}
# Verify all required secrets exist before rendering any clips.
render_env = env_for(['PEXELS_API_KEY'] if pexels else [])
secrets_for = {'youtube':['YOUTUBE_TOKEN_JSON'], 'facebook':['FB_PAGE_ID','FB_PAGE_TOKEN']}
publish_env = {t:{**env_for(secrets_for[t]), 'YOUTUBE_VISIBILITY':visibility} for t in targets}
for slot in range(1, count+1):
    job = batch / f'{slot:02d}'
    if not job.exists():
        with tempfile.TemporaryDirectory() as work:
            cmd = [sys.executable,'studio.py','--output',work,'--seed',day.replace('-','')+str(slot)]
            if voice: cmd.append('--voice')
            if pexels: cmd.append('--pexels')
            generated = Path(subprocess.check_output(cmd,cwd=root,env=render_env,text=True).strip())
            # Expose a finished job only after both files have been copied.
            staging = Path(tempfile.mkdtemp(dir=batch))
            try:
                shutil.copytree(generated,staging,dirs_exist_ok=True)
                staging.rename(job)
            finally:
                if staging.exists(): shutil.rmtree(staging)
    print(f'{slot}/{count}: {job}',flush=True)
    for target in targets:
        subprocess.run([sys.executable,'publish.py','upload',str(job),target],cwd=root,env=publish_env[target],check=True)
display(Video(str(job/'lesson.mp4'),embed=True))